# Разметка страниц

Проверка шага, который до детектора строк находит страницы на фото и вырезает их ровными.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from omegaconf import OmegaConf
from PIL import Image, ImageOps

from data import MEAN, STD, Pages
from infer import PageFinder, draw
from pages import cut

cfg = OmegaConf.load("config.yaml")

## Примеры глазами модели

Красным — `paper` (вся бумага), синим — `core` (страница без каймы). На развороте должно быть два синих пятна.

In [ ]:
ds = Pages(cfg, train=True)
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax in axes.flat:
    x, y, _ = ds[np.random.randint(len(ds))]
    img = (x.numpy().transpose(1, 2, 0) * STD + MEAN).clip(0, 1)
    paper, core = (np.kron(m, np.ones((cfg.data.stride, cfg.data.stride)))[..., None] for m in y.numpy())
    tint = img * (1 - 0.4 * paper) + 0.4 * paper * np.array([1, 0, 0])
    tint = tint * (1 - 0.4 * core) + 0.4 * core * np.array([0, 0, 1])
    ax.imshow(tint)
    ax.axis("off")
plt.tight_layout()
plt.show()

## Обученная модель на своих фото

Впишите пути к своим снимкам.

In [ ]:
finder = PageFinder(cfg, "outputs/best.pt")
paths = ["/path/to/photo1.jpg", "/path/to/photo2.jpg"]

In [ ]:
for path in paths:
    photo = ImageOps.exif_transpose(Image.open(path)).convert("RGB")
    quads = finder.quads(photo)
    fig, axes = plt.subplots(1, 1 + len(quads), figsize=(6 * (1 + len(quads)), 6))
    axes = np.atleast_1d(axes)
    axes[0].imshow(draw(photo, quads))
    axes[0].set_title(f"{path}: страниц {len(quads)}")
    for k, (ax, quad) in enumerate(zip(axes[1:], quads), 1):
        page = cut(photo, quad)
        ax.imshow(page.image)
        ax.set_title(f"страница {k}: {page.image.width} x {page.image.height}")
    for ax in axes:
        ax.axis("off")
    plt.show()